# ABPT Anchor V1 — Google Colab Training Notebook

Этот notebook готовит Colab/Google Connect запуск для `ABPTAnchorV1`.

Что он делает:
1. монтирует Google Drive (опционально)
2. клонирует репозиторий или использует уже существующую папку
3. проверяет GPU и окружение
4. запускает короткий `world-like` warmup на `tiny_shakespeare`
5. запускает короткий synthetic anchor run на `anchor-synthetic`
6. сохраняет history и генерирует markdown reports

По умолчанию notebook **ничего не публикует наружу** — все артефакты лежат в рабочей папке Colab или на Drive.


In [ ]:
# @title 1. Mount Google Drive (optional)
USE_DRIVE = True
DRIVE_ROOT = '/content/drive/MyDrive'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted:', DRIVE_ROOT)
else:
    print('Drive mount skipped')


In [ ]:
# @title 2. Clone/open repo
from pathlib import Path
import os

# Если у тебя есть Git URL репозитория — вставь сюда.
REPO_URL = ''  # e.g. 'https://github.com/<user>/<repo>.git'
REPO_DIR = Path('/content/ABPT')

if REPO_DIR.exists():
    print('Repo already exists:', REPO_DIR)
elif REPO_URL:
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    raise RuntimeError('Set REPO_URL or upload/copy the repo to /content/ABPT before continuing.')

%cd /content/ABPT
print('Working dir:', REPO_DIR)


In [ ]:
# @title 3. Install dependencies
import sys
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q torch einops pytest


In [ ]:
# @title 4. Runtime check
import torch, platform, os, json
info = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'devices': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'cwd': os.getcwd(),
}
print(json.dumps(info, indent=2))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)


In [ ]:
# @title 5. Optional smoke tests
RUN_TESTS = False
if RUN_TESTS:
    !pytest -q tests/test_abpt_anchor_v1.py tests/test_train_eval.py tests/test_anchor_synthetic.py
else:
    print('Smoke tests skipped')


## World-like warmup

Это не настоящая Phase 1 на тысячи шагов, а короткий Colab-friendly запуск для проверки пайплайна.

Если хочешь сделать реальный warmup, увеличь `WORLD_STEPS` до 5000+ и используй GPU runtime.


In [ ]:
# @title 6. World-like warmup on tiny_shakespeare
from pathlib import Path

WORLD_PRESET = 'toy'
WORLD_STEPS = 200
WORLD_HISTORY = Path('archive/colab_world_history.json')
WORLD_REPORT = Path('docs/research/colab_world_report.md')

!python train.py --preset {WORLD_PRESET} --stage anchor --dataset shakespeare --device {DEVICE} --steps {WORLD_STEPS} --history_path {WORLD_HISTORY}
!python scripts/generate_anchor_training_report.py {WORLD_HISTORY} --output {WORLD_REPORT}
print(WORLD_REPORT)


In [ ]:
# @title 7. Show world report
from pathlib import Path
from IPython.display import Markdown, display
report_text = Path('docs/research/colab_world_report.md').read_text(encoding='utf-8')
display(Markdown(report_text))


## Synthetic anchor run

Это текущий theory-aligned шаг: проверить, как Anchor V1 ведёт себя на `anchor-synthetic` внутри train loop.

Если `proposal_influence` остаётся нулевым, это значит, что следующая работа должна идти в explicit proposal/revision supervision, а не просто в большее число шагов.


In [ ]:
# @title 8. Synthetic anchor training run
from pathlib import Path

SYN_PRESET = 'toy'
SYN_STEPS = 200
SYN_HISTORY = Path('archive/colab_anchor_synth_history.json')
SYN_REPORT = Path('docs/research/colab_anchor_synth_report.md')

!python train.py --preset {SYN_PRESET} --stage anchor --dataset anchor-synthetic --device {DEVICE} --steps {SYN_STEPS} --history_path {SYN_HISTORY}
!python scripts/generate_anchor_training_report.py {SYN_HISTORY} --output {SYN_REPORT}
print(SYN_REPORT)


In [ ]:
# @title 9. Show synthetic report
from pathlib import Path
from IPython.display import Markdown, display
report_text = Path('docs/research/colab_anchor_synth_report.md').read_text(encoding='utf-8')
display(Markdown(report_text))


In [ ]:
# @title 10. Save artifacts to Drive (optional)
from pathlib import Path
import shutil

SAVE_TO_DRIVE = USE_DRIVE
TARGET_DIR = Path(DRIVE_ROOT) / 'ABPT_colab_runs'
ARTIFACTS = [
    Path('archive/colab_world_history.json'),
    Path('archive/colab_anchor_synth_history.json'),
    Path('docs/research/colab_world_report.md'),
    Path('docs/research/colab_anchor_synth_report.md'),
]

if SAVE_TO_DRIVE:
    TARGET_DIR.mkdir(parents=True, exist_ok=True)
    for path in ARTIFACTS:
        if path.exists():
            dst = TARGET_DIR / path.name
            shutil.copy2(path, dst)
            print('Saved:', dst)
else:
    print('Saving to Drive skipped')
